In [1]:
!pip install sqlalchemy==1.3.9

     ---------------------------------------- 0.0/6.0 MB ? eta -:--:--
     ---------------------------------------- 0.0/6.0 MB ? eta -:--:--
     ----- ---------------------------------- 0.8/6.0 MB 6.0 MB/s eta 0:00:01
     ------------ --------------------------- 1.8/6.0 MB 5.3 MB/s eta 0:00:01
     ---------------------- ----------------- 3.4/6.0 MB 6.0 MB/s eta 0:00:01
     --------------------------------- ------ 5.0/6.0 MB 6.5 MB/s eta 0:00:01
     ---------------------------------------- 6.0/6.0 MB 6.8 MB/s  0:00:01
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for sqlalchemy: filename=sqlalchemy-1.3.9-cp313-cp313-win_amd64.whl size=1143040 sha256=d954746a55592d7221d68e634838efc25aad7

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
aext-project-filebrowser-server 4.20.0 requires sqlalchemy>=2.0.29, but you have sqlalchemy 1.3.9 which is incompatible.
alembic 1.18.3 requires SQLAlchemy>=1.4.23, but you have sqlalchemy 1.3.9 which is incompatible.


In [2]:
!pip install ipython-sql
!pip install ipython-sql prettytable

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------  2.1/2.1 MB 14.5 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 12.4 MB/s  0:00:00

   ---------------------------------------- 0/5 [ipython-genutils]
   ---------------------------------------- 0/5 [ipython-genutils]
   -------- ------------------------------- 1/5 [sqlparse]
   -------- ------------------------------- 1/5 [sqlparse]
   -------- ------------------------------- 1/5 [sqlparse]
   -------- ------------------------------- 1/5 [sqlparse]
   -------- ------------------------------- 1/5 [sqlparse]
  Attempting uninstall: sqlalchemy
   -------- ------------------------------- 1/5 [sqlparse]
    Found existing installation: SQLAlchemy 1.3.9
   -------- ------------------------------- 1/5 [sqlparse]
   ---------------- ----------------------- 2/5 [sqlalchemy]
    Uninstalling SQLAlchemy-1.3.9:
   ---------------- ----------------------- 2/5 [sq

In [3]:
%load_ext sql

In [4]:
import csv, sqlite3
import prettytable
prettytable.DEFAULT = 'DEFAULT'
con = sqlite3.connect("my_data1.db")
cur = con.cursor()

In [5]:
%sql sqlite:///my_data1.db

In [6]:
import pandas as pd

In [7]:
df = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv")
df.to_sql("SPACEXTBL", con, if_exists='replace', index=False,method="multi")

101

In [8]:
query = """
SELECT DISTINCT Launch_Site
FROM SPACEXTBL
"""

In [9]:
pd.read_sql(query, con)

,Launch_Site
0,CCAFS LC-40
1,VAFB SLC-4E
2,KSC LC-39A
3,CCAFS SLC-40


In [10]:
query = """
SELECT *
FROM SPACEXTBL
WHERE Launch_Site LIKE 'CCA%'
LIMIT 5
"""

In [11]:
pd.read_sql(query, con)

,Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
0,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
1,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of...",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
3,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
4,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


In [12]:
query = """
SELECT AVG(Payload_Mass__kg_) AS avg_payload_mass
FROM SPACEXTBL
WHERE Booster_Version = 'F9 v1.1'
"""

In [13]:
pd.read_sql(query, con)

,avg_payload_mass
0,2928.4


In [14]:
query = """
SELECT MIN(Date) AS first_successful_ground_pad
FROM SPACEXTBL
WHERE Landing_Outcome = 'Success (ground pad)'
"""

In [15]:
pd.read_sql(query, con)

,first_successful_ground_pad
0,2015-12-22


In [16]:
query = """
SELECT DISTINCT Booster_Version
FROM SPACEXTBL
WHERE Landing_Outcome = 'Success (drone ship)'
  AND Payload_Mass__kg_ > 4000
  AND Payload_Mass__kg_ < 6000
"""

In [17]:
pd.read_sql(query, con)

,Booster_Version
0,F9 FT B1022
1,F9 FT B1026
2,F9 FT B1021.2
3,F9 FT B1031.2


In [18]:
query = """
SELECT 
    CASE 
        WHEN Mission_Outcome LIKE 'Success%' THEN 'Success'
        ELSE 'Failure'
    END AS outcome_category,
    COUNT(*) AS total_count
FROM SPACEXTBL
GROUP BY outcome_category
"""

In [19]:
pd.read_sql(query, con)

,outcome_category,total_count
0,Failure,1
1,Success,100


In [20]:
query = """
SELECT DISTINCT Booster_Version
FROM SPACEXTBL
WHERE Payload_Mass__kg_ = (
    SELECT MAX(Payload_Mass__kg_)
    FROM SPACEXTBL
)
"""

In [21]:
pd.read_sql(query, con)

,Booster_Version
0,F9 B5 B1048.4
1,F9 B5 B1049.4
2,F9 B5 B1051.3
3,F9 B5 B1056.4
4,F9 B5 B1048.5
5,F9 B5 B1051.4
6,F9 B5 B1049.5
7,F9 B5 B1060.2
8,F9 B5 B1058.3
9,F9 B5 B1051.6


In [22]:
query = """
SELECT 
    CASE strftime('%m', Date)
        WHEN '01' THEN 'January'
        WHEN '02' THEN 'February'
        WHEN '03' THEN 'March'
        WHEN '04' THEN 'April'
        WHEN '05' THEN 'May'
        WHEN '06' THEN 'June'
        WHEN '07' THEN 'July'
        WHEN '08' THEN 'August'
        WHEN '09' THEN 'September'
        WHEN '10' THEN 'October'
        WHEN '11' THEN 'November'
        WHEN '12' THEN 'December'
    END AS month_name,
    Landing_Outcome,
    Booster_Version,
    Launch_Site
FROM SPACEXTBL
WHERE strftime('%Y', Date) = '2015'
  AND Landing_Outcome LIKE 'Failure%drone ship%'
"""

In [23]:
pd.read_sql(query, con)

,month_name,Landing_Outcome,Booster_Version,Launch_Site
0,January,Failure (drone ship),F9 v1.1 B1012,CCAFS LC-40
1,April,Failure (drone ship),F9 v1.1 B1015,CCAFS LC-40


In [24]:
query = """
SELECT 
    Landing_Outcome,
    COUNT(*) AS outcome_count,
    RANK() OVER (ORDER BY COUNT(*) DESC) AS rank
FROM SPACEXTBL
WHERE Date BETWEEN '2010-06-04' AND '2017-03-20'
GROUP BY Landing_Outcome
"""

In [25]:
pd.read_sql(query, con)

,Landing_Outcome,outcome_count,rank
0,No attempt,10,1
1,Success (drone ship),5,2
2,Failure (drone ship),5,2
3,Success (ground pad),3,4
4,Controlled (ocean),3,4
5,Uncontrolled (ocean),2,6
6,Failure (parachute),2,6
7,Precluded (drone ship),1,8
